# 04 · Validate — developability gate + cross-strain breadth

**Standard slot:** *validate (in silico).* **For Project 14 the core science is BREADTH + developability:**
does each top VHH hold across a strain panel, and is it developable? Report the **worst-case** strain (D3 pt 2).

## Setup paths

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Developability gate (teaching heuristics — swap in real tools)

Filter survivors on the developability **proxies** (lower `tap_score`, higher `humanness`). These are
CLEARLY-LABELED teaching heuristics, NOT validated TAP/CamSol/Hu-mAb — replace before any conclusion.

In [ ]:
import pandas as pd
ranked = pd.read_csv("results/ranked.csv")
designs = pd.read_csv("results/designs.csv")[["design_id", "tap_score", "humanness", "camsol_like", "cdr3"]]
df = ranked.merge(designs, on="design_id", how="left")
TAP_MAX, HUMAN_MIN = 6.0, 0.60   # EXAMPLE teaching thresholds — calibrate with real tools
dev_ok = df[(df["tap_score"] <= TAP_MAX) & (df["humanness"] >= HUMAN_MIN)]
print(f"developability gate (tap<={TAP_MAX}, humanness>={HUMAN_MIN}): {len(dev_ok)}/{len(df)} pass (SYNTHETIC)")
dev_ok[["design_id", "layers_passed", "pae_interaction", "tap_score", "humanness"]].head(8)

## 2 · Cross-strain breadth

Model each top candidate against the strain panel; record per-strain `pae_interaction`; rank by the
**worst-case** strain. The panel is **EXAMPLE_DATA** (mock); use your real verified strains.

In [ ]:
import antibody_tools as ab, numpy as np
top = (dev_ok if len(dev_ok) else df).head(10)
STRAINS = ["H1", "H3", "H5", "H7", "InfB"]   # EXAMPLE panel — verify/replace
rows = []
for r in top.itertuples():
    seq = designs.set_index('design_id').loc[r.design_id, 'cdr3'] if False else str(r.sequence)
    prof = ab.breadth_across_strains(str(r.sequence), STRAINS, tool="mock")
    prof["design_id"] = r.design_id
    prof["worst_case_pae"] = max(v for v in prof.values() if isinstance(v, (int, float)))
    rows.append(prof)
breadth = pd.DataFrame(rows).set_index("design_id").sort_values("worst_case_pae")
breadth.to_csv("results/breadth.csv")
print("breadth profile (SYNTHETIC) — broadest (lowest worst-case pae) first:")
breadth

In [ ]:
import matplotlib.pyplot as plt
panel = [c for c in breadth.columns if c != "worst_case_pae"]
fig, ax = plt.subplots(figsize=(7, 4))
for did, row in breadth.iterrows():
    ax.plot(panel, [row[c] for c in panel], marker="o", alpha=0.6, label=str(did)[:18])
ax.axhline(12, ls="--", c="k", lw=0.8, label="antibody cutoff 12")
ax.set_ylabel("pae_interaction (lower = better)"); ax.set_xlabel("strain")
ax.set_title("Cross-strain breadth (EXAMPLE_DATA / mock)")
plt.xticks(rotation=20); plt.tight_layout(); plt.savefig("results/proj14_breadth.png", dpi=150); plt.show()

## D3 (part 2) checklist
- [ ] Developability gate applied (real tools swapped in for any reported conclusion).
- [ ] Cross-strain breadth table + figure; **worst-case** strain reported per candidate.
- [ ] RFantibody vs BoltzGen head-to-head (hit rate, CDR geometry, developability) `[extension]`.
- [ ] Failure-mode notes: narrow/escape-prone or liability-laden designs.

**Next:** `05_validation_plan.ipynb` — the yeast-display screen + neutralization plan.